<a href="https://colab.research.google.com/github/sanyamsh7/Agentic-RAG-Pipeline-From-Scratch/blob/main/Module2_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 2 - TEXT CHUNKING

CORE CONCEPTS I COVERED :
1. The WHY of Chunking (3 major problems it solves):

    * Context window limits
    * Retrieval precision
    * API costs


2. Naive vs. Smart Chunking:

    See how naive chunking breaks text Learn why RecursiveCharacterTextSplitter is superior


3. Critical Parameters:

    * chunk_size: How big each piece should be
    * chunk_overlap: Why overlapping is crucial for context


4. Hands-on Examples:

    * Visual comparison of different chunk sizes
    * Different overlap values and their impact
    * Complete chunking pipeline


6. Chunk Analysis:

    * Statistics on chunks
    * Distribution visualization
    * Quality checks

In [ ]:
# installing libraries
!pip install langchain langchain-community langchain-text-splitters -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)
from langchain_community.document_loaders import PyPDFLoader
import textwrap

In [ ]:
# to understand why we need chunking ? let's create a sample document for this
sample_document = """
The Transformer architecture revolutionized natural language processing.
It was introduced in the paper "Attention is All You Need" by Vaswani et al.
The key innovation was the self-attention mechanism.

Self-attention allows the model to weigh the importance of different words
in a sentence when processing each word. This is more powerful than
previous sequential models like RNNs and LSTMs.

The architecture consists of an encoder and decoder. The encoder processes
the input sequence, while the decoder generates the output sequence.
Both use multi-head attention mechanisms.

Transformers are now the foundation of modern LLMs like GPT, BERT, and Claude.
They enable models to process long contexts and understand complex relationships
between words across large distances in text.
"""

In [ ]:
print(len(sample_document))

790


 Understanding context window limits -
 Language models have a MAXIMUM input size (context window):
- GPT-3.5: ~ 4,096 tokens (~16,000 characters)
- GPT-4: ~ 8,192 tokens (~32,000 characters)  
- Claude 3: ~ 200,000 tokens (~800,000 characters)

If we have a 500-page PDF (500,000+ characters), we CAN'T feed
the entire document to the model at once!

Understanding retrieval precision -
Even if you COULD fit everything, you shouldn't!

Imagine asking: "How does self-attention work?"

❌ BAD: Return entire 15-page document
   - Model gets overwhelmed with irrelevant info
   - Slower processing
   - Harder to find the answer

✅ GOOD: Return only the 2-3 paragraphs about self-attention
   - Focused, relevant context
   - Faster and cheaper
   - Better answers

Understanding API costs - API calls are charged per token:
- Sending 10,000 tokens costs more than 500 tokens
- Chunking = Only send relevant pieces = Lower cost

In [ ]:
# naive chunking - wrong way!
def naive_chunk(text: str, chunk_size: int):
  """
  just splitting N characters. Don't do this!
  """
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])
  return chunks

In [ ]:
# trying naive chunking
naive_chunks = naive_chunk(sample_document, chunk_size = 100)

for i, chunk in enumerate(naive_chunks[:5], 1):
    print(f"\nChunk {i}:")
    print(f"'{chunk}'")
    print("-" * 70)


Chunk 1:
'
The Transformer architecture revolutionized natural language processing.
It was introduced in the p'
----------------------------------------------------------------------

Chunk 2:
'aper "Attention is All You Need" by Vaswani et al.
The key innovation was the self-attention mechani'
----------------------------------------------------------------------

Chunk 3:
'sm.

Self-attention allows the model to weigh the importance of different words
in a sentence when p'
----------------------------------------------------------------------

Chunk 4:
'rocessing each word. This is more powerful than
previous sequential models like RNNs and LSTMs.

The'
----------------------------------------------------------------------

Chunk 5:
' architecture consists of an encoder and decoder. The encoder processes
the input sequence, while th'
----------------------------------------------------------------------


Problem with naive chunking -
1. Splits mid-sentence: "Self-attention allows the model to weigh the impo..."
2. Breaks words
3. No context preservation: Chunk 2 starts randomly
4. Semantic meaning destroyed: Related sentences get separated

**RecursiveCharacterTextSplitter** - This is the BEST general-purpose chunker. Here's why:

It tries to split on separators in this ORDER:
1. Double newline (\\n\\n) - Paragraph breaks
2. Single newline (\\n) - Line breaks  
3. Space ( ) - Word boundaries
4. Character - Only as last resort

This preserves semantic structure!

In [ ]:
# smart chunking - right way! - RecursiveCharacterTextSplitter
# create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200, # max char per chunk
    chunk_overlap = 50, # characters to overlap between chunks
    length_function = len, # how to measure length ( can use token counter as well)
    is_separator_regex= False
)

# splitting our sample
smart_chunks = text_splitter.split_text(sample_document)
print(f"created {len(smart_chunks)} chunks")

created 6 chunks


In [ ]:
for i, chunk in enumerate(smart_chunks, 1):
    print(f"\nChunk {i} ({len(chunk)} chars :")
    print(f"'{chunk.strip()}'")
    print("-"*70)


Chunk 1 (149 chars :
'The Transformer architecture revolutionized natural language processing.
It was introduced in the paper "Attention is All You Need" by Vaswani et al.'
----------------------------------------------------------------------

Chunk 2 (52 chars :
'The key innovation was the self-attention mechanism.'
----------------------------------------------------------------------

Chunk 3 (190 chars :
'Self-attention allows the model to weigh the importance of different words
in a sentence when processing each word. This is more powerful than
previous sequential models like RNNs and LSTMs.'
----------------------------------------------------------------------

Chunk 4 (185 chars :
'The architecture consists of an encoder and decoder. The encoder processes
the input sequence, while the decoder generates the output sequence.
Both use multi-head attention mechanisms.'
----------------------------------------------------------------------

Chunk 5 (159 chars :
'Transformers are n

Notice the improvement here:
1. splits at paragraph boundaries.
2. complete sentences in each chunks.
3. no broken words.
4. semantic meaning preserved.
5. related content stays together.

Understanding chunk overlap -
WHY OVERLAP ?

Imagine this sentence is split across two chunks:

- Chunk 1: "...The transformer uses self-attention."

- Chunk 2: "It allows the model to focus on relevant words."

❌ Without overlap: "It" in Chunk 2 is ambiguous!

✅ With overlap: Chunk 2 includes "...self-attention. It allows..."

THE OVERLAP PROVIDED CONTEXT.


In [ ]:
# understanding with different overlap
overlaps = [0,20,50]

for overlap in overlaps:
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = 100,
      chunk_overlap = overlap,
  )
  chunks = splitter.split_text(sample_document)
  print(chunks)
  print("-" * 70)

['The Transformer architecture revolutionized natural language processing.', 'It was introduced in the paper "Attention is All You Need" by Vaswani et al.', 'The key innovation was the self-attention mechanism.', 'Self-attention allows the model to weigh the importance of different words', 'in a sentence when processing each word. This is more powerful than', 'previous sequential models like RNNs and LSTMs.', 'The architecture consists of an encoder and decoder. The encoder processes', 'the input sequence, while the decoder generates the output sequence.', 'Both use multi-head attention mechanisms.', 'Transformers are now the foundation of modern LLMs like GPT, BERT, and Claude.', 'They enable models to process long contexts and understand complex relationships', 'between words across large distances in text.']
----------------------------------------------------------------------
['The Transformer architecture revolutionized natural language processing.', 'It was introduced in the pap

📌 OVERLAP GUIDELINES:
- Too little (0-10%): Chunks lose context
- Good (10-20%): Balances context and redundancy  
- Too much (50%+): Wasteful, duplicate info

RULE OF THUMB: overlap = 10-20% of chunk_size

Choosing the right chunk size - There's no perfect chunk size! it depends on:
1. Content type:
    - Dense technical docs: 500-1000 chars ( detailed info )
    - General articles: 1000-2000 chars ( broader context )
    - FAQ/Short answers: 200-500 chars ( focused )

2. Use Case:
    - Precise fact extraction: Smaller chunks ( 500 )
    - Broader context needed: Larger chunks ( 1500 )
    - Mixed queries: Medium chunks ( 800-1000 )

3. Model:
    - Embedding Model: 512 usually tokens max
    - LLM context: More flexible but cost matters


In [ ]:
# let's check different chunk size on our document
chunk_size = [100,200,500]
for size in chunk_size:
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = size,
      chunk_overlap = int(size*0.15) # for 15% overlap
  )
  chunks = splitter.split_text(sample_document)
  avg_length = sum(len(c) for c in chunks)/len(chunks)
  print(f"\nChunk Size = {size}:")
  print(f"  → {len(chunks)} chunks created")
  print(f"  → Average chunk length: {avg_length:.0f} chars")
  print(f"  → Overlap: {int(size * 0.15)} chars")


Chunk Size = 100:
  → 12 chunks created
  → Average chunk length: 64 chars
  → Overlap: 15 chars

Chunk Size = 200:
  → 6 chunks created
  → Average chunk length: 130 chars
  → Overlap: 30 chars

Chunk Size = 500:
  → 2 chunks created
  → Average chunk length: 393 chars
  → Overlap: 75 chars


In [ ]:
# now chunk the original pdf documents
# creating a sample document data ( if pdf not present )
from langchain_core.documents import Document

# Simulate loaded PDF pages
sample_pages = [
    Document(
        page_content=sample_document,
        metadata={"source": "sample.pdf", "page": 1}
    ),
    Document(
        page_content=sample_document,  # Using same text for demo
        metadata={"source": "sample.pdf", "page": 2}
    ),
]

print(f"📄 Starting with {len(sample_pages)} document pages")

📄 Starting with 2 document pages


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [ ]:
# Split documents (not just text!)
chunks = text_splitter.split_documents(sample_pages)

In [ ]:
sample_document

'\nThe Transformer architecture revolutionized natural language processing.\nIt was introduced in the paper "Attention is All You Need" by Vaswani et al.\nThe key innovation was the self-attention mechanism.\n\nSelf-attention allows the model to weigh the importance of different words\nin a sentence when processing each word. This is more powerful than\nprevious sequential models like RNNs and LSTMs.\n\nThe architecture consists of an encoder and decoder. The encoder processes\nthe input sequence, while the decoder generates the output sequence.\nBoth use multi-head attention mechanisms.\n\nTransformers are now the foundation of modern LLMs like GPT, BERT, and Claude.\nThey enable models to process long contexts and understand complex relationships\nbetween words across large distances in text.\n'

In [ ]:
chunks

[Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='The Transformer architecture revolutionized natural language processing.\nIt was introduced in the paper "Attention is All You Need" by Vaswani et al.\nThe key innovation was the self-attention mechanism.'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='Self-attention allows the model to weigh the importance of different words\nin a sentence when processing each word. This is more powerful than\nprevious sequential models like RNNs and LSTMs.'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='The architecture consists of an encoder and decoder. The encoder processes\nthe input sequence, while the decoder generates the output sequence.\nBoth use multi-head attention mechanisms.'),
 Document(metadata={'source': 'sample.pdf', 'page': 1}, page_content='Transformers are now the foundation of modern LLMs like GPT, BERT, and Claude.\nThey enable models to process long contexts a

In [ ]:
# after chunking let's examine the chunks
for i, chunk in enumerate(chunks[:4], 1):
    print(f"\nChunk {i}:")
    print(f"  Metadata: {chunk.metadata}")
    print(f"  Content ({len(chunk.page_content)} chars): {chunk.page_content[:100]}...")
    print("-" * 70)


Chunk 1:
  Metadata: {'source': 'sample.pdf', 'page': 1}
  Content (202 chars): The Transformer architecture revolutionized natural language processing.
It was introduced in the pa...
----------------------------------------------------------------------

Chunk 2:
  Metadata: {'source': 'sample.pdf', 'page': 1}
  Content (190 chars): Self-attention allows the model to weigh the importance of different words
in a sentence when proces...
----------------------------------------------------------------------

Chunk 3:
  Metadata: {'source': 'sample.pdf', 'page': 1}
  Content (185 chars): The architecture consists of an encoder and decoder. The encoder processes
the input sequence, while...
----------------------------------------------------------------------

Chunk 4:
  Metadata: {'source': 'sample.pdf', 'page': 1}
  Content (205 chars): Transformers are now the foundation of modern LLMs like GPT, BERT, and Claude.
They enable models to...
-----------------------------------------------

Metadata is preservedd!

Each chunk knows:
- which file it came from ( source )
- which page it was on ( page )

Preserving metadata is crucial for CITATIONS.

In [ ]:
# let's visualize our chunks
def analyze_chunks(chunks):
    """
    Analyze chunk statistics to understand the distribution.
    """
    lengths = [len(chunk.page_content) for chunk in chunks]

    print(f"Total chunks: {len(chunks)}")
    print(f"Average length: {sum(lengths) / len(lengths):.0f} characters")
    print(f"Min length: {min(lengths)} characters")
    print(f"Max length: {max(lengths)} characters")
    print()

    # Simple text-based histogram
    print("Length distribution:")
    bins = [0, 100, 200, 300, 400, 500]
    for i in range(len(bins) - 1):
        count = sum(1 for l in lengths if bins[i] <= l < bins[i+1])
        bar = "█" * (count * 2)
        print(f"  {bins[i]:3d}-{bins[i+1]:3d}: {bar} ({count})")

    # Check for very short chunks (might indicate problems)
    very_short = [l for l in lengths if l < 50]
    if very_short:
        print(f"\n⚠️  Warning: {len(very_short)} chunks are very short (<50 chars)")
        print("   Consider adjusting chunk_size or check your separators")

analyze_chunks(chunks)

Total chunks: 8
Average length: 196 characters
Min length: 185 characters
Max length: 205 characters

Length distribution:
    0-100:  (0)
  100-200: ████████ (4)
  200-300: ████████ (4)
  300-400:  (0)
  400-500:  (0)


RecursiveCharacterTextSplitter is great for MOST cases, but others exist:

1. CharacterTextSplitter
   - Splits on a single separator only
   - Use when: You have consistent structure (e.g., always \\n\\n)

2. TokenTextSplitter  
   - Splits based on TOKEN count (not characters)
   - Use when: Working with token-limited models
   - More accurate for cost/limit calculations

3. MarkdownTextSplitter
   - Preserves markdown structure (headers, lists)
   - Use when: Processing markdown documentation

4. CodeTextSplitter
   - Respects code structure (functions, classes)
   - Use when: Processing source code

For 80% of RAG use cases: RecursiveCharacterTextSplitter is perfectly fine.

## Practical example - complete flow

In [ ]:
def complete_chunking_pipeline(documents, chunk_size=500, chunk_overlap=80):
  print(len(documents), "documents loaded")

  splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunks = splitter.split_documents(documents)

  print(f"Split into {len(chunks)} chunks")

  analyze_chunks(chunks)
  return chunks

In [ ]:
final_chunks = complete_chunking_pipeline(
    sample_pages,
    chunk_size=300,
    chunk_overlap=50
)

2 documents loaded
Split into 8 chunks
Total chunks: 8
Average length: 196 characters
Min length: 185 characters
Max length: 205 characters

Length distribution:
    0-100:  (0)
  100-200: ████████ (4)
  200-300: ████████ (4)
  300-400:  (0)
  400-500:  (0)


Key Insights:     
🎯 GOLDEN RULES:

├─ chunk_size: 500-800 chars (good starting point)

├─ chunk_overlap: 15-20% of chunk_size  

├─ Always preserve metadata (source, page number)

└─ Use RecursiveCharacterTextSplitter for most cases